In [1]:
from diffusers import AutoPipelineForInpainting
import torch
import os, sys
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm import tqdm
import yaml
import tempfile
import cv2

# Add the project root directory to sys.path
project_root = os.path.abspath('..')  # Adjust this path if needed
sys.path.append(project_root)

from utils import load_images_and_combine, load_images
from utils import mask_on_image, dilate_mask, plot_grid

2024-08-23 09:40:34.370563: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-23 09:40:35.983684: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-23 09:40:36.188993: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-08-23 09:40:37.437574: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-23 09:40:41.564548: W tensorflow/compiler/tf2

[08/23/24 09:40:58] WARNING  Your inference package version 0.12.1 is out of date! Please upgrade to ]8;id=624251;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=823490;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py#35\35]8;;\
                             version 0.16.2 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

In [2]:
init_model = 'lama' #@param ['lama', 'sd1.5', 'paint_by_example', 'ldm']
port = 4242
device = 'cuda' #@param ['cuda', 'cpu', 'mps']

cmd = f"iopaint start --model {init_model} --host 0.0.0.0 --port {port} --device {device}"
print(cmd)
# start subprocess using cmd
#subprocess.run(cmd, shell=True)

iopaint start --model lama --host 0.0.0.0 --port 4242 --device cuda


In [3]:
# Load the YAML file into a dictionary
dataset_yaml_file = '/workspace/home/data/datasets.yaml'
with open(dataset_yaml_file, 'r') as file:
    datasets = yaml.safe_load(file)

In [4]:
# Load images
dataset_key ='ISIC2019'# 'Pascal_VOC07' #
dataset = datasets[dataset_key]
images_confounder, images_no_confounder = load_images(dir_path=f"/workspace/home/data/{dataset['dataset_dir']}/raw", confounder_dir=dataset['confounder_dir'], no_confounder_dir=dataset['no_confounder_dir'])
segmentation_masks_confounder, _ = load_images(dir_path=f"/workspace/home/data/{dataset['dataset_dir']}/seg", confounder_dir=dataset['confounder_dir'], no_confounder_dir=dataset['no_confounder_dir'])

output_dir = f"/workspace/home/outputfiles/{dataset['dataset_dir']}/inpainting/iopaint/{init_model}"

output_dir_confounder = os.path.join(output_dir, dataset['confounder_dir'])
output_dir_no_confounder = os.path.join(output_dir, dataset['no_confounder_dir'])

# Create output directories if they don't exist
os.makedirs(output_dir_confounder, exist_ok=True)
os.makedirs(output_dir_no_confounder, exist_ok=True)

In [5]:
prompt = "part of skin in homogenous skin color"
# "part of skin"
#"remove the confounder"
# ""
#"remove colorful patch"
#"lesion on skin"

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


for i, (image_dict, mask_dict) in enumerate(tqdm(zip(images_confounder, segmentation_masks_confounder))):
  image = image_dict['image']
  meta_img = image_dict['meta']
  img_url = meta_img['file_path']

  mask = mask_dict['image']
  meta_mask = mask_dict['meta']
  mask_url = meta_mask['file_path']

  #image = load_image(img_url)
  #mask_image = load_image(mask_url)

  dilated_mask = dilate_mask(Image.open(mask_url), kernel_size=5, iterations=5)

  with tempfile.TemporaryDirectory() as tmpdirname:

    # Define the path for the temporary mask file
    mask_path = os.path.join(tmpdirname, 'mask.jpg')
    # Save the dilated mask to the temporary file
    cv2.imwrite(mask_path, dilated_mask)

    !iopaint run --model={init_model} --device={device} \
    --image={img_url} \
    --mask={mask_path} \
    --output={output_dir_confounder}



for image_dict in images_no_confounder:
  image = image_dict['image']
  meta_img = image_dict['meta']
  img_url = meta_img['file_path']

  # save image as PIL image
  img = Image.open(img_url)
  img.save(f"{output_dir_no_confounder}/{meta_img['file_name']}")


0it [00:00, ?it/s]

2024-08-23 09:41:07.283 | INFO     | iopaint.runtime:setup_model_dir:82 - Model directory: /root/.cache
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/usr/local/lib/python3.10/site-packages/iopaint/model/ldm.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast()
/usr/local/lib/python3.10/si

1it [00:18, 18.22s/it]

2024-08-23 09:41:25.121 | INFO     | iopaint.runtime:setup_model_dir:82 - Model directory: /root/.cache
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/usr/local/lib/python3.10/site-packages/iopaint/model/ldm.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast()
/usr/local/lib/python3.10/si

2it [00:29, 14.05s/it]

...

2024-08-23 09:43:04.330 | INFO     | iopaint.runtime:setup_model_dir:82 - Model directory: /root/.cache
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/usr/local/lib/python3.10/site-packages/iopaint/model/ldm.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast()
/usr/local/lib/python3.10/si

11it [02:08, 11.11s/it]

2024-08-23 09:43:15.317 | INFO     | iopaint.runtime:setup_model_dir:82 - Model directory: /root/.cache
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/usr/local/lib/python3.10/site-packages/iopaint/model/ldm.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast()
/usr/local/lib/python3.10/si

12it [02:17, 10.39s/it]

2024-08-23 09:43:24.074 | INFO     | iopaint.runtime:setup_model_dir:82 - Model directory: /root/.cache
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.10/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")
/usr/local/lib/python3.10/site-packages/iopaint/model/ldm.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast()
/usr/local/lib/python3.10/si